# Memory
> LangGraph通过持久化检查点（Checkpointing）机制解决这一问题，支持多轮会话状态保存与恢复，还能扩展用于错误恢复、人工介入工作流等复杂场景



In [1]:

from typing import Annotated
from langchain_openai import ChatOpenAI
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os
load_dotenv()

True

In [2]:
class State(TypedDict):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
)

def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)


## 记忆存储
* 内存存储--开发阶段
* 支持Sqlite等数据库存储

In [3]:
memory = MemorySaver()
# 编译图的时候指定 checkpointer
graph = graph_builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "1"}}  # 线程ID为"1"的会话
print(config)


{'configurable': {'thread_id': '1'}}


In [4]:
while True:
    q = input("用户: ")
    if q == "exit":
        break
    state = {
        "messages": [HumanMessage(content=q)]
    }
    out = graph.invoke(state, config)
    final_msg = out["messages"][-1]
    print(">>> 答案:", final_msg.content)

>>> 答案: 你好呀！✨ 很高兴见到你！今天过得怎么样呀？希望你度过了愉快的一天。我随时准备好陪你聊天、帮你解决问题，或者就这样轻松愉快地闲聊一会儿。有什么想跟我分享的吗？ 🌟
>>> 答案: 嗨！😊 看来你可能遇到了什么有趣的事，或者有什么想法想和我分享呢？我在这里听着呢～ 无论是开心的事想一起庆祝，还是有点小烦恼需要倾诉，都可以告诉我哦！让我们开始一段愉快的对话吧！ 🌈💬


## 长对话处理
由于可能导致的过多的token浪费，和最长上下文的溢出问题，必须对长对话进行处理
langgraph的处理方式：
* 动态裁剪：根据上下文的长度，动态裁剪掉旧的消息
* 总结摘要：对旧的消息进行总结，生成摘要；触发条件：当消息数或 token 数超过阈值时，触发 LLM 进行总结


## 实现LangGraph 长对话处理的完整示例
动态裁剪 + 总结摘要

In [5]:
from typing_extensions import Annotated
from typing import List

from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages.utils import trim_messages
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage

from langchain_openai import ChatOpenAI

In [6]:
class ConversationState(TypedDict):
    # messages: 用 Annotated 包装，支持动态裁剪
    messages: Annotated[List[BaseMessage], trim_messages(
        # 保留的最大 token 数
        max_tokens=800,
        # 当超过 max_tokens 时，裁掉最早的消息
        strategy="last"
    )]
    # 存储对话的摘要
    summary: str = ""

llm = ChatOpenAI(
    model=os.getenv("MODEL_NAME"),
    base_url=os.getenv("BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
    temperature=0,
)

def agent_node(state: ConversationState):
    # 构造完整上下文：先加摘要（若有），再加所有历史消息
    context = []
    # 1. 若有摘要，先添加摘要
    if state["summary"]:
        print(f"执行agent_node有摘要: {state["summary"]}")
        context.append(SystemMessage(content=f"对话历史摘要：{state["summary"]}"))
    # 2. 添加所有历史消息
    context.extend(state["messages"])   

    # 3. 调用 LLM 生成回复
    response = llm.invoke(context)
    
    # 4. 返回新的 AI 消息（LangGraph 会自动追加到 state.messages）
    return {
        "messages": [AIMessage(content=response.content)]
    }

def summarizer_node(state: ConversationState):
    print("进入summarizer_node")
    # 先计算当前 messages 的总 Token 数
    def count_tokens(messages: List[BaseMessage]) -> int:
        # 可以按照1token = 4字符计算
        return sum(len(msg.content) // 4 for msg in messages)
    
    # 若消息数超过 3 条，或总 Token 数超过 1000 个
    if len(state["messages"]) > 3 or count_tokens(state["messages"]) > 1000:
        # 调用 LLM 生成摘要
        summary_prompt = [
            SystemMessage(content="请将以下对话内容总结成简洁摘要（100字内），保留关键问题和结论："),
            # 合并摘要+消息（确保摘要的连续性）
            HumanMessage(content=f"历史摘要：{state["summary"]}\n当前对话：\n" + "\n".join([m.content for m in state["summary"]]))
        ]
        summary = llm.invoke(summary_prompt)
        return {
            "summary": summary.content,
            # 把原始消息清空，只保留摘要
            "messages": [SystemMessage(content=f"对话摘要：{summary.content}")]
        }
    # 不满足条件时，返回空字典（状态不变）
    return {}

def get_user_input() -> str:
    print("\n" + "-"*50)
    user_input = input("请输入你的问题（输入 'exit' 结束对话）：")
    # 处理空输入（用户按回车）
    if not user_input.strip():
        print("提示：输入不能为空，请重新输入！")
        return get_user_input()  # 递归获取，直到输入有效
    return user_input.strip()

def should_continue(state: ConversationState) -> str:
    # 获取用户输入
    user_input = get_user_input()
    if user_input.lower() == 'exit':
        print("感谢使用，再见！")
        return "end"
    # 把用户输入添加到状态的 messages 中
    state["messages"].append(HumanMessage(content=user_input))
    print(f"用户输入：{user_input}")
    print("正在处理中...")

    return "summarizer"

builder = StateGraph(ConversationState)
builder.add_node("agent", agent_node)
builder.add_node("summarizer", summarizer_node)
builder.add_node("end", lambda state: state)    # 结束节点，返回当前状态

builder.add_edge(START, "agent")
builder.add_conditional_edges(
    "agent",
    should_continue,
    {
        "summarizer": "summarizer",
        "end": "end",
    },
)
builder.add_edge("summarizer", "agent")


graph = builder.compile()


In [7]:
state = {
    "messages": [HumanMessage(content="你好，我叫星星")],
    "summary": ""
}


# 执行图流程
result = graph.invoke(state)

# 打印最终结果
print("\n" + "-"*50)
print("最终对话摘要：", result["summary"])
print("最终消息列表：")
for i, msg in enumerate(result["messages"], 1):
    role = "系统" if isinstance(msg, SystemMessage) else "AI" if isinstance(msg, AIMessage) else "用户"
    print(f"{i}. [{role}] {msg.content}")


--------------------------------------------------
用户输入：q
正在处理中...
进入summarizer_node

--------------------------------------------------
感谢使用，再见！

--------------------------------------------------
最终对话摘要： 
最终消息列表：
1. [AI] 嘿嘿，简短有力的“q”～是想来一场说走就走的冒险，还是想问点什么呀？🤔✨ 随时等你发号施令哦～😊
